In [12]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

from src.schemas import ORDERS_SCHEMA

In [2]:
spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

In [3]:
spark.sql("""
    SELECT
        current_catalog() AS catalog,
        current_schema() AS schema
""").show()


+---------+-------+
|  catalog| schema|
+---------+-------+
|workspace|default|
+---------+-------+



In [4]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

DataFrame[]

In [6]:
spark.sql("SHOW SCHEMAS IN workspace").show()

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|information_schema|
+------------------+

None


In [7]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files
""")

DataFrame[]

In [10]:
spark.sql("""
    SHOW VOLUMES IN workspace.bronze
""").show(truncate=False)

+--------+-----------+
|database|volume_name|
+--------+-----------+
|bronze  |raw_files  |
+--------+-----------+



In [14]:
raw_orders_path = "/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv"

raw_orders = (
    spark.read
    .option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(raw_orders_path)
)

raw_orders.printSchema()
print("Row count: ", raw_orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Row count:  99441


In [16]:
bronze_orders = (
    raw_orders
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_source_file_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_entity", F.lit("orders"))
)

bronze_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _source_file_modified_at: timestamp (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_entity: string (nullable = false)



In [17]:
bronze_orders.select(
    "order_id",
    "_source_file",
    "_source_file_modified_at",
    "_ingested_at",
    "_source_entity",
).show(5, truncate=False)

+--------------------------------+-----------------------------------------------------------------+------------------------+--------------------------+--------------+
|order_id                        |_source_file                                                     |_source_file_modified_at|_ingested_at              |_source_entity|
+--------------------------------+-----------------------------------------------------------------+------------------------+--------------------------+--------------+
|e481f51cbdc54678b7cc49136f2d6af7|dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-10 11:57:09     |2026-08-10 12:07:36.865974|orders        |
|53cdb2fc8bc7dce0b6741e2150273451|dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-10 11:57:09     |2026-08-10 12:07:36.865974|orders        |
|47770eb9100c2d0c44946d9cf07ec65d|dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-10 11:57:09     |2026-08-10 12:07:36.865974|orders  

In [18]:
(
    bronze_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.orders")
)

In [21]:
spark.sql("""
    DESCRIBE DETAIL workspace.bronze.orders
""").show(truncate=False)

+------+------------------------------------+-----------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                   |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                   

In [23]:
spark.sql("""
    DESCRIBE HISTORY workspace.bronze.orders
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+------------------------------------+-----------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName                     |operation                        |operationParameters                                                                                                                        

In [24]:
demo = spark.createDataFrame(
    [
        (1, "first"),
        (2, "second"),
        (3, "third"),
    ],
    ["id", "value"]
)

(
    demo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [26]:
spark.sql("""
    DESCRIBE HISTORY workspace.bronze.delta_history_demo
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+------------------------------------+-----------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName                     |operation                        |operationParameters                                                                                                                               

In [27]:
new_row = spark.createDataFrame(
    [(4, "fourth")],
    ["id", "value"]
)

(
    new_row.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [30]:
spark.sql("""
    DESCRIBE HISTORY workspace.bronze.delta_history_demo
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+------------------------------------+-----------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName                     |operation                        |operationParameters                                                                                                                               

In [31]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    VERSION AS OF 0
    ORDER BY id
""").show()

+---+------+
| id| value|
+---+------+
|  1| first|
|  2|second|
|  3| third|
+---+------+



In [32]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show()

+---+------+
| id| value|
+---+------+
|  1| first|
|  2|second|
|  3| third|
|  4|fourth|
+---+------+



In [33]:
bad_schema_df = spark.createDataFrame(
    [
        (5, "fifth", "unexpected")
    ],
    ["id", "value", "extra_column"]
)

(
    bad_schema_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 26c30384-a6db-46fa-b4dc-57178974ace5).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)


Data schema:
root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)
 |-- extra_column: string (nullable = true)


- Table ACLs are enabled in this cluster, so automatic schema migration is not allowed. Please use the ALTER TABLE command for changing the schema.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisExceptionWithSubErrors
	at com.databricks.sql.trans

In [34]:
spark.sql("""
    ALTER TABLE workspace.bronze.delta_history_demo
    ADD COLUMNS (extra_column STRING)
""")

DataFrame[]

In [35]:
spark.table(
    "workspace.bronze.delta_history_demo"
).printSchema()

root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)
 |-- extra_column: string (nullable = true)



In [36]:
(
    bad_schema_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [38]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show()

+---+------+------------+
| id| value|extra_column|
+---+------+------------+
|  1| first|        NULL|
|  2|second|        NULL|
|  3| third|        NULL|
|  4|fourth|        NULL|
|  5| fifth|  unexpected|
+---+------+------------+



In [40]:
spark.sql("""
    SELECT
        version,
        timestamp,
        operation,
        readVersion,
        operationMetrics
    FROM (
        DESCRIBE HISTORY workspace.bronze.delta_history_demo
    )
    ORDER BY version DESC
""").show(truncate=False)

+-------+-------------------+---------------------------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |readVersion|operationMetrics                                                                                                                       |
+-------+-------------------+---------------------------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------+
|3      |2026-08-10 12:29:58|WRITE                            |2          |{numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1324}                                                                            |
|2      |2026-08-10 12:26:54|ADD COLUMNS                      |1          |{}                                                                           

In [42]:
replacement_df = spark.createDataFrame(
    [
        (100, "replacement_a", "new"),
        (200, "replacement_b", "new"),
    ],
    ["id", "value", "extra_column"]
)
replacement_df.show()

+---+-------------+------------+
| id|        value|extra_column|
+---+-------------+------------+
|100|replacement_a|         new|
|200|replacement_b|         new|
+---+-------------+------------+



In [43]:
(
    replacement_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [45]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show()

+---+-------------+------------+
| id|        value|extra_column|
+---+-------------+------------+
|100|replacement_a|         new|
|200|replacement_b|         new|
+---+-------------+------------+



In [47]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    VERSION AS OF 3
    ORDER BY id
""").show()

+---+------+------------+
| id| value|extra_column|
+---+------+------------+
|  1| first|        NULL|
|  2|second|        NULL|
|  3| third|        NULL|
|  4|fourth|        NULL|
|  5| fifth|  unexpected|
+---+------+------------+



In [50]:
updates_df = spark.createDataFrame(
    [
        (100, "replacement_a_updated", "updated"),
        (300, "replacement_c", "new"),
    ],
    ["id", "value", "extra_column"]
)

updates_df.show()

+---+--------------------+------------+
| id|               value|extra_column|
+---+--------------------+------------+
|100|replacement_a_upd...|     updated|
|300|       replacement_c|         new|
+---+--------------------+------------+



In [51]:
updates_df.createOrReplaceTempView("updates")

In [52]:
spark.sql("""
    MERGE INTO workspace.bronze.delta_history_demo AS target
    USING updates AS source
    ON target.id = source.id

    WHEN MATCHED THEN
        UPDATE SET *

    WHEN NOT MATCHED THEN
        INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [54]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show(truncate=False)

+---+---------------------+------------+
|id |value                |extra_column|
+---+---------------------+------------+
|100|replacement_a_updated|updated     |
|200|replacement_b        |new         |
|300|replacement_c        |new         |
+---+---------------------+------------+



In [56]:
spark.sql("""
    SELECT
        version,
        operation,
        readVersion,
        operationMetrics
    FROM (
        DESCRIBE HISTORY workspace.bronze.delta_history_demo
    )
    ORDER BY version DESC
""").show(truncate=False)

+-------+---------------------------------+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation                        |readVersion|operationMetrics                                                                                                                                                                                                                   

In [60]:
(
    spark.sql("""
        DESCRIBE TABLE EXTENDED workspace.bronze.delta_history_demo
    """)
    .filter("col_name = 'Predictive Optimization'")
    .show(truncate=False)
)


+-----------------------+---------------------------------------------------------+-------+
|col_name               |data_type                                                |comment|
+-----------------------+---------------------------------------------------------+-------+
|Predictive Optimization|ENABLE (inherited from METASTORE metastore_aws_us_east_2)|       |
+-----------------------+---------------------------------------------------------+-------+

None
